# Q2 training on Colab

Runtime → Change runtime type → **GPU** (T4, L4 or A100), then run the cells in order.

Checkpoints are saved to Google Drive after **every epoch**. If the session disconnects,
reconnect, run cells 1–4 again, then rerun the training cell: each run resumes where it stopped.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Mount Google Drive (checkpoints and logs live here, so they survive disconnects)
from google.colab import drive
drive.mount('/content/drive')
WORK = '/content/drive/MyDrive/DL4STG_PA2'
import os; os.makedirs(WORK, exist_ok=True)

In [ ]:
# 3. Get the latest code from GitHub
import os
if os.path.isdir('/content/DL4TSG_PA2'):
    !git -C /content/DL4TSG_PA2 pull -q
else:
    !git clone -q https://github.com/sikanderraheem21/DL4TSG_PA2.git /content/DL4TSG_PA2
%cd /content/DL4TSG_PA2
!git log --oneline -1
!pip -q install opencv-python imageio

In [ ]:
# 4. Moving MNIST: downloaded once, cached on Drive, copied to fast local disk each session
import os, shutil
local = '/content/DL4TSG_PA2/data/MovingMNIST'
cached = f'{WORK}/data/MovingMNIST'
if os.path.isdir(cached) and not os.path.isdir(local):
    shutil.copytree(cached, local)
from q2.data import load_videos
print(load_videos('data').shape)          # expect torch.Size([10000, 20, 1, 64, 64])
if not os.path.isdir(cached):
    shutil.copytree(local, cached)

## Training runs

Edit `RUNS` to choose what to train. Each line is one model and seed, 30 epochs.
The first epoch's time tells you how long the rest will take.

**Phase 2:** model A only. Models B, C and D are trained in Phase 4, *after* the
Q2.11 prediction has been committed.

In [ ]:
# 5. Train
RUNS = [('A', 0), ('A', 1)]
for model, seed in RUNS:
    !python -m q2.train --model {model} --seed {seed} --ckpt-dir {WORK}/checkpoints --log-dir {WORK}/train_logs

In [ ]:
# 6. Download the final checkpoints and logs (small: about 2 MB per model)
from google.colab import files
!rm -f /content/q2_results.zip && cd {WORK} && zip -q -r /content/q2_results.zip checkpoints/*_s?.pt train_logs
files.download('/content/q2_results.zip')